# Portfolio Backtest
Run strategy backtests directly via the trading engine — no API, no frontend.

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import date
import pandas as pd

from trading_engine.data.yfinance_loader import YFinanceLoader
from trading_engine.strategy.buy_and_hold import BuyAndHold
from trading_engine.strategy.ma_crossover import MACrossover
from trading_engine.portfolio.simulation import run_portfolio
from trading_engine.types import Portfolio

loader = YFinanceLoader()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SYMBOLS         = ["MSFT"]   # ← change this (can be a list)
START           = date(2015, 1, 1)
END             = date.today()
INITIAL_CAPITAL = 10_000

# Load prices for all symbols
prices = {sym: loader.load(sym, START, END) for sym in SYMBOLS}
for sym, p in prices.items():
    print(f"{sym}: {len(p.data)} bars  ({p.data.index[0].date()} → {p.data.index[-1].date()})")

## Buy & Hold

In [ ]:
portfolio = Portfolio(
    initial_capital=INITIAL_CAPITAL,
    strategy=BuyAndHold(weight=1.0),
    max_leverage=1.0,
)

result = run_portfolio(portfolio=portfolio, symbols=SYMBOLS, prices=prices)

equity = pd.Series(result.equity_curve)
print(f"Total return:  {(equity.iloc[-1]/equity.iloc[0] - 1)*100:.2f}%")
print(f"Final NAV:     ${equity.iloc[-1]:,.2f}")
print(f"Trades:        {len(result.trades)}")
print()
print("Equity curve (last 5):")
print(equity.tail(5).to_string())

## MA Crossover

In [ ]:
FAST = 10    # ← change this
SLOW = 50    # ← change this
MA   = "SMA" # SMA | EMA | WMA

portfolio = Portfolio(
    initial_capital=INITIAL_CAPITAL,
    strategy=MACrossover(
        fast_length=FAST,
        slow_length=SLOW,
        fast_ma_type=MA,
        slow_ma_type=MA,
    ),
    max_leverage=1.0,
)

result_ma = run_portfolio(portfolio=portfolio, symbols=SYMBOLS, prices=prices)

equity_ma = pd.Series(result_ma.equity_curve)
print(f"Strategy: MA Crossover {FAST}/{SLOW} {MA}")
print(f"Total return:  {(equity_ma.iloc[-1]/equity_ma.iloc[0] - 1)*100:.2f}%")
print(f"Final NAV:     ${equity_ma.iloc[-1]:,.2f}")
print(f"Trades:        {len(result_ma.trades)}")

## Inspect trades

In [ ]:
trades = result_ma.trades

rows = []
for t in trades:
    rows.append({
        "Symbol":      t.symbol,
        "Direction":   t.direction,
        "Entry Date":  t.entry_date,
        "Entry Price": f"${t.entry_price:.2f}",
        "Exit Date":   t.exit_date or "OPEN",
        "Exit Price":  f"${t.exit_price:.2f}" if t.exit_price else "—",
        "Return %":    f"{t.return_pct:+.2f}%" if t.return_pct is not None else "—",
        "Hold Days":   t.holding_days or "—",
        "MAE %":       f"{t.mae_pct:.2f}%" if t.mae_pct is not None else "—",
        "MFE %":       f"{t.mfe_pct:+.2f}%" if t.mfe_pct is not None else "—",
    })

df_trades = pd.DataFrame(rows)
print(f"Total trades: {len(df_trades)}")
df_trades

## Quick metrics from trades

In [ ]:
closed = [t for t in trades if t.return_pct is not None and t.exit_date is not None]
wins   = [t for t in closed if t.return_pct > 0]
losses = [t for t in closed if t.return_pct <= 0]

def pct(val): return f"{val*100:.2f}%"

win_rate = len(wins) / len(closed) if closed else 0
avg_win  = sum(t.return_pct for t in wins)  / len(wins)  if wins   else 0
avg_loss = sum(t.return_pct for t in losses)/ len(losses) if losses else 0
best     = max((t.return_pct for t in closed), default=0)
worst    = min((t.return_pct for t in closed), default=0)

# Max drawdown from equity curve
eq = equity_ma.sort_index()
peak = eq.expanding().max()
dd   = (eq - peak) / peak
max_dd = dd.min()

print(f"Closed trades:         {len(closed)}")
print(f"Win rate:              {pct(win_rate)}")
print(f"Avg win:               {pct(avg_win)}")
print(f"Avg loss:              {pct(avg_loss)}")
print(f"Best trade:            {pct(best)}")
print(f"Worst trade:           {pct(worst)}")
print(f"Max drawdown:          {pct(max_dd)}")

## Strategy vs Buy & Hold comparison

In [ ]:
eq_bah = pd.Series(result.equity_curve).sort_index()
eq_ma  = pd.Series(result_ma.equity_curve).sort_index()

def max_dd(equity):
    peak = equity.expanding().max()
    return ((equity - peak) / peak).min()

ret_bah = (eq_bah.iloc[-1] / eq_bah.iloc[0] - 1) * 100
ret_ma  = (eq_ma.iloc[-1]  / eq_ma.iloc[0]  - 1) * 100
dd_bah  = max_dd(eq_bah) * 100
dd_ma   = max_dd(eq_ma)  * 100

comparison = pd.DataFrame({
    "Buy & Hold": {
        "Total Return": f"{ret_bah:+.2f}%",
        "Max Drawdown": f"{dd_bah:.2f}%",
        "Final NAV":    f"${eq_bah.iloc[-1]:,.2f}",
        "Trades":       len(result.trades),
    },
    f"MA {FAST}/{SLOW} {MA}": {
        "Total Return": f"{ret_ma:+.2f}%",
        "Max Drawdown": f"{dd_ma:.2f}%",
        "Final NAV":    f"${eq_ma.iloc[-1]:,.2f}",
        "Trades":       len(result_ma.trades),
    },
})

comparison